# Buscar los enlaces de YouTube de las canciones

Completa la columna `youtube_url` de `canciones_metadata.csv` usando la API oficial de YouTube.

**Antes de empezar**
1. En el proyecto, corre `python src/crear_metadata.py` para generar `data/canciones_metadata.csv`.
2. Consigue una clave de la API de YouTube Data v3 (Google Cloud Console → habilitar *YouTube Data API v3* → crear credencial *Clave de API*).
3. En Colab, guarda la clave en **Secretos** (ícono de llave, a la izquierda) con el nombre `YOUTUBE_API_KEY` y activa el acceso del cuaderno.

Cada búsqueda gasta 100 unidades de la cuota diaria (10 000 por defecto), así que caben unas 100 por día. Si tienes más canciones, vuelve a correr el cuaderno al día siguiente con el CSV que descargaste: continúa donde quedó.

In [ ]:
# Título: Importar librerías
import html
import json
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd
from google.colab import files

In [ ]:
# Título: Subir el CSV de metadatos
subido = files.upload()  # elige canciones_metadata.csv
nombre = next(iter(subido))
df = pd.read_csv(nombre, dtype=str, encoding="utf-8-sig").fillna("")
print(f"{len(df)} canciones | {(df['youtube_url'] != '').sum()} con enlace | {(df['youtube_url'] == '').sum()} pendientes")

In [ ]:
# Título: Leer la clave de la API
try:
    from google.colab import userdata
    CLAVE = userdata.get("YOUTUBE_API_KEY")
except Exception:
    from getpass import getpass
    CLAVE = getpass("Pega tu clave de la API de YouTube: ")

In [ ]:
# Título: Definir la búsqueda en YouTube
class CuotaAgotada(Exception):
    pass


def buscar(consulta, clave):
    params = {"part": "snippet", "q": consulta, "type": "video", "videoEmbeddable": "true",
              "maxResults": 1, "key": clave}
    url = "https://www.googleapis.com/youtube/v3/search?" + urllib.parse.urlencode(params)
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            items = json.load(r).get("items", [])
    except urllib.error.HTTPError as e:
        cuerpo = e.read().decode("utf-8", errors="ignore")
        if e.code == 403 and "quota" in cuerpo.lower():
            raise CuotaAgotada from e
        raise
    if not items:
        return None
    it = items[0]
    return {"url": "https://www.youtube.com/watch?v=" + it["id"]["videoId"],
            "titulo": html.unescape(it["snippet"]["title"]),
            "canal": html.unescape(it["snippet"]["channelTitle"])}

In [ ]:
# Título: Buscar los enlaces pendientes (máximo 95 por corrida)
MAXIMO = 95
pendientes = df.index[(df["youtube_url"] == "") & (df["artista"] != "")].tolist()
print(f"Se buscarán {min(len(pendientes), MAXIMO)} de {len(pendientes)} pendientes")
hechas = 0
for i in pendientes[:MAXIMO]:
    fila = df.loc[i]
    try:
        r = buscar(f"{fila['artista']} {fila['titulo']}".strip(), CLAVE)
    except CuotaAgotada:
        print("Cuota diaria agotada: continúa mañana con el CSV descargado.")
        break
    hechas += 1
    if r is None:
        print("  [sin resultado]", fila["archivo"])
        continue
    df.loc[i, ["youtube_url", "youtube_titulo", "youtube_canal", "estado"]] = [r["url"], r["titulo"], r["canal"], "auto"]
    print("  [auto]", fila["archivo"], "->", r["titulo"])
print(f"Búsquedas hechas: {hechas}")

In [ ]:
# Título: Revisar lo encontrado
# Compara youtube_titulo con el título real; si es un cover o una versión en vivo, corrígelo en el CSV.
df[df["estado"] == "auto"][["archivo", "youtube_titulo", "youtube_canal", "youtube_url"]].head(30)

In [ ]:
# Título: Descargar el CSV actualizado
df.to_csv("canciones_metadata.csv", index=False, encoding="utf-8-sig")
files.download("canciones_metadata.csv")
# Copia el archivo descargado a data/canciones_metadata.csv del proyecto (reemplaza el anterior).